In [8]:
#Chargement et extraction de contenu d’un fichier PDF avec LangChain (PyPDFLoader)
from langchain_community.document_loaders import PyPDFLoader
from IPython.display import display, Markdown

loader = PyPDFLoader("acmecorp-employee-handbook-1.pdf")
data = loader.load()

contenu = "\n\n".join(doc.page_content for doc in data)

display(Markdown(contenu))



Employee Handbook
Non-Disclosure Agreement (NDA) Policy
Employees must protect confidential information belonging to the company, its clients, and partners.
This includes, but is not limited to, product roadmaps, customer data, internal communications,
proprietary algorithms, financial information, and unreleased features. Confidential information may not
be shared with unauthorized individuals inside or outside the organization. These obligations continue
after employment ends.
Workplace Conduct Policy
Employees must maintain a respectful, professional environment free from harassment, discrimination,
and intimidation. All employees are expected to follow organizational values, collaborate effectively,
and communicate constructively. Disruptive behavior, verbal abuse, or misuse of company systems is
prohibited. Violations may result in disciplinary action.
Paid Time Off (PTO) Policy
Full■time employees accrue PTO according to the following schedule:  0–1 years of service: 10 days
per year (0.833 days per month)  1–3 years of service: 15 days per year (1.25 days per month)  3+
years of service: 20 days per year (1.67 days per month) PTO may be used for vacation, personal
needs, or illness. Requests should be submitted in advance through the HR system unless related to
an emergency. Employees may carry over up to 5 unused PTO days per calendar year. Extended
absences exceeding 5 consecutive business days require manager approval.
Travel & Expense Policy
Employees may be reimbursed for reasonable and necessary expenses incurred during approved
business travel. This includes transportation, lodging, meals, and incidental expenses within
established limits. Receipts must be submitted within 14 days of travel. First-class travel, personal
expenses, and non-business activities are not reimbursable. Employees should exercise good
judgment and cost-effective decision-making when traveling on behalf of the company.

In [13]:
#Segmentation de texte pour préparation au RAG avec LangChain
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(data)
print(len(all_splits))
for text in all_splits:
   print(f"Begin ---: {text} :----End")

3
Begin ---: page_content='Employee Handbook
Non-Disclosure Agreement (NDA) Policy
Employees must protect confidential information belonging to the company, its clients, and partners.
This includes, but is not limited to, product roadmaps, customer data, internal communications,
proprietary algorithms, financial information, and unreleased features. Confidential information may not
be shared with unauthorized individuals inside or outside the organization. These obligations continue
after employment ends.
Workplace Conduct Policy
Employees must maintain a respectful, professional environment free from harassment, discrimination,
and intimidation. All employees are expected to follow organizational values, collaborate effectively,
and communicate constructively. Disruptive behavior, verbal abuse, or misuse of company systems is
prohibited. Violations may result in disciplinary action.
Paid Time Off (PTO) Policy
Full■time employees accrue PTO according to the following schedule:  0–1 ye

In [14]:
#Transformation des chunks en vecteurs sémantiques avec un modèle Hugging Face : Génération d’embeddings textuels
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

#Création d’une base vectorielle en mémoire pour stockage et recherche d’embeddings
from langchain_core.vectorstores import InMemoryVectorStore
vector_store = InMemoryVectorStore(embeddings)

#Indexation des documents dans la base vectorielle pour recherche sémantique
ids = vector_store.add_documents(documents=all_splits)

#Recherche sémantique dans une base vectorielle pour retrouver les informations pertinentes 
results = vector_store.similarity_search("How many days of vacation does an employee get in their first year?")
print(results[0])

C:\Users\User\AppData\Local\Temp\ipykernel_30132\3192351257.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6163.96it/s]


page_content='prohibited. Violations may result in disciplinary action.
Paid Time Off (PTO) Policy
Full■time employees accrue PTO according to the following schedule:  0–1 years of service: 10 days
per year (0.833 days per month)  1–3 years of service: 15 days per year (1.25 days per month)  3+
years of service: 20 days per year (1.67 days per month) PTO may be used for vacation, personal
needs, or illness. Requests should be submitted in advance through the HR system unless related to
an emergency. Employees may carry over up to 5 unused PTO days per calendar year. Extended
absences exceeding 5 consecutive business days require manager approval.
Travel & Expense Policy
Employees may be reimbursed for reasonable and necessary expenses incurred during approved
business travel. This includes transportation, lodging, meals, and incidental expenses within
established limits. Receipts must be submitted within 14 days of travel. First-class travel, personal' metadata={'producer': 'ReportL

In [17]:
#Agent RAG
from langchain.tools import tool
@tool
def search_handbook(query: str) -> str:
    """
    
    """
    results = vector_store.similarity_search(query)
    return results[0].page_content

from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_ollama import ChatOllama

# Initialiser le modèle Ollama
model = ChatOllama(
model="llama3.2", # ou mistral, gemma, etc.
temperature=0
)
agent = create_agent(
model=model,
tools=[search_handbook],
system_prompt="You are a helpful agent that can search the employee handbook for information.")
response = agent.invoke({"messages": [HumanMessage(content="How many days of vacation does an employee get in their first year?")]})
print(response['messages'][-1].content)

According to the handbook, in an employee's first year of employment, they accrue 10 days of paid time off (PTO) per year, which translates to approximately 0.833 days per month.


In [20]:
#Connexion à une base de données SQL (SQLite) avec LangChain
from langchain_community.utilities import SQLDatabase
db = SQLDatabase.from_uri("sqlite:///Chinook.db")

from langchain.tools import tool
@tool
def sql_query(query: str) -> str:
    """Obtain information from the database using SQL queries"""
    try:
        print(f"Executing SQL query: {query}")
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

sql_query.invoke("SELECT * FROM Artist LIMIT 10")



Executing SQL query: SELECT * FROM Artist LIMIT 10


"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains'), (6, 'Antônio Carlos Jobim'), (7, 'Apocalyptica'), (8, 'Audioslave'), (9, 'BackBeat'), (10, 'Billy Cobham')]"

In [23]:
#Création d’un agent LLM pour interroger une base SQL
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_ollama import ChatOllama
# Initialiser le modèle Ollama
model = ChatOllama(
model="llama3.2", # ou mistral, gemma, etc.
)
system_prompt = """You are a SQL expert.
Rules:
- Only use sql_query tool
- The sql_query tool takes a SQL query as input and returns the result of the
query.
- Only use available columns
- If information does not exist, say so
- Do not guess
- you have to return the results in a human readable format, do not return raw
SQL results or a sql query.
Database schema:
Table Artist:
- ArtistId
- Name
"""
agent = create_agent(model=model,tools=[sql_query],system_prompt=system_prompt)
#Interrogation d’un agent SQL via langage naturel et récupération des résultats
question = HumanMessage(content="Give me the first 5 artists in the database")
response = agent.invoke({"messages": [question]})
print(response['messages'][-1].content)

Executing SQL query: SELECT Name FROM Artist LIMIT 5
The first 5 artists in the database are:

1. AC/DC
2. Accept
3. Aerosmith
4. Alanis Morissette
5. Alice In Chains
